In [1]:
import random
import pandas as pd

In [2]:
def _parse_occurrence(occ):
    """Convertit un element de la liste occurrences en (k, is_A)."""
    if isinstance(occ, str) and occ.upper().endswith('A'):
        return int(occ[:-1]), True
    return int(occ), False

In [12]:
def _simulate_process(N, p, occurrences_parsed, trigger_soul, trigger_blank):
    """Simule un processus complet et renvoie X (total de cartes conservees)."""
    deck = ['B'] * p + ['W'] * (N - p)
    random.shuffle(deck)
    discard = []
 
    trigger_deck = ['Bl'] * trigger_soul + ['Rd'] * trigger_blank
    random.shuffle(trigger_deck)
    trigger_discard = []
 
    clock_zone = []
    total_kept = 0
 
    for base_k, is_A in occurrences_parsed:
        k = base_k
 
        # --- resolution trigger pour occurrence A ---
        if is_A:
            if not trigger_deck and trigger_discard:
                trigger_deck = trigger_discard[:]
                trigger_discard = []
                random.shuffle(trigger_deck)
            if trigger_deck:
                card = trigger_deck.pop()
                trigger_discard.append(card)
                if card == 'Bl':
                    k += 1
 
        # --- resolution du tirage principal de l'occurrence ---
        drawn_this_occurrence = []
        occurrence_failed = False
        halted = False
        draws_done = 0
 
        while draws_done < k:
            if len(deck) == 0:
                # regle de refresh
                if not discard:
                    halted = True
                    break
                new_deck = discard[:]
                discard = []
                random.shuffle(new_deck)
                clock_card = new_deck.pop()  # carte au hasard -> clock zone
                clock_zone.append(clock_card)
                total_kept += 1  # compte dans X, blanche ou noire
                deck = new_deck
                if len(deck) == 1:
                    halted = True
                    break
                continue
 
            card = deck.pop()
            drawn_this_occurrence.append(card)
            draws_done += 1
            if card == 'B':
                occurrence_failed = True
                break
 
        if halted:
            discard.extend(drawn_this_occurrence)
            break
 
        if occurrence_failed:
            discard.extend(drawn_this_occurrence)
        else:
            # succes : toutes les cartes tirees sont blanches -> conservees
            total_kept += len(drawn_this_occurrence)
            clock_zone.extend(drawn_this_occurrence)
 
            # regle de level-up (par paquets de 7)
            while len(clock_zone) >= 7:
                batch = clock_zone[:7]
                clock_zone = clock_zone[7:]
                white_idx = next((i for i, c in enumerate(batch) if c == 'W'), None)
                if white_idx is not None:
                    batch.pop(white_idx)
                else:
                    batch.pop(0)
                discard.extend(batch)
 
    return total_kept

In [15]:
def run_simulation(N_list, p_list, occurrences, trigger_soul, trigger_blank, n_trials=20000):
    """
    Lance la simulation Monte Carlo et renvoie un DataFrame pandas.
 
    Parametres
    ----------
    N_list : list[int]        tailles de decks a tester
    p_list : list[int]        nombres de climax a tester
    occurrences : list        dégâts associés au finisher
    trigger_soul : int        nombre de cartes avec trigger dans le deck trigger
    trigger_blank : int       nombre de cartes sans trigger dans le deck trigger
    n_trials : int            nombre d'essais Monte Carlo par combinaison [N,p]
 
    Retour
    ------
    pandas.DataFrame avec une ligne par combinaison [N,p] et une colonne
    par seuil X_min contenant P(X >= X_min). Utiliser df.to_csv(chemin, index=False)
    pour exporter le resultat.
    """
    occurrences_parsed = [_parse_occurrence(o) for o in occurrences]
    max_possible = sum(k + (1 if is_A else 0) for k, is_A in occurrences_parsed)
 
    rows = []
    for N in N_list:
        for p in p_list:
            totals = [
                _simulate_process(N, p, occurrences_parsed, trigger_soul, trigger_blank)
                for _ in range(n_trials)
            ]
            row = {'N': N, 'p': p}
            for x in range(0, max_possible + 1):
                row[f'X_min={x}'] = round(sum(1 for t in totals if t >= x) / n_trials, 4)
            rows.append(row)
 
    return pd.DataFrame(rows)

In [16]:
table = run_simulation(
        N_list=[20,25,30],
        p_list=[4,6,8],
        occurrences=[4,'3A',4,'3A',4,'3A'],
        trigger_soul = 15,
        trigger_blank = 35,
        n_trials=10000,
        )

table.to_csv('resultats_simulation.csv', index=False)
table

,N,p,X_min=0,X_min=1,X_min=2,X_min=3,X_min=4,X_min=5,X_min=6,X_min=7,...,X_min=15,X_min=16,X_min=17,X_min=18,X_min=19,X_min=20,X_min=21,X_min=22,X_min=23,X_min=24
0,20,4,1.0,1.0000,1.0000,1.0000,1.0000,0.9995,0.9978,0.9345,...,0.0079,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
1,20,6,1.0,0.9365,0.9365,0.9365,0.7382,0.4750,0.4750,0.4002,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
2,20,8,1.0,0.6715,0.6715,0.6715,0.4122,0.1334,0.1334,0.0998,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
3,25,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9726,...,0.1137,0.0322,0.0180,0.0120,0.0025,0.0001,0.0000,0.0000,0.0000,0.0
4,25,6,1.0,0.9864,0.9864,0.9864,0.9040,0.7788,0.7788,0.7044,...,0.0074,0.0012,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
5,25,8,1.0,0.8771,0.8771,0.8771,0.6734,0.4121,0.4121,0.3432,...,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
6,30,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9893,...,0.2732,0.1214,0.0881,0.0672,0.0215,0.0038,0.0023,0.0008,0.0001,0.0
7,30,6,1.0,0.9956,0.9956,0.9956,0.9551,0.9010,0.9010,0.8512,...,0.0517,0.0134,0.0069,0.0045,0.0012,0.0000,0.0000,0.0000,0.0000,0.0
8,30,8,1.0,0.9521,0.9521,0.9521,0.8299,0.6592,0.6592,0.5892,...,0.0057,0.0008,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
